In [ ]:
#!pip install gurobipy
#!pip install numpy
#!pip install scikit-learn
#!pip install matplotlib
#!pip install pandas

# Truck Rebalancing Optimization

This notebook demonstrates the full optimization pipeline used to compute
weekly truck rebalancing routes for a bike sharing system.

Pipeline overview:

1. Load strategic station objectives (Pareto frontiers)
2. Merge geographic information
3. Prepare optimization parameters
4. Solve routing models
5. Export and visualize routes

The optimization is based on mixed-integer programming solved with **Gurobi**.

In [ ]:
# ⚠ Important notice
# This notebook uses Gurobi, a commercial optimization solver.
# A license is required to run the full optimization problem.
#
# You can obtain:
# - a free trial license: https://www.gurobi.com/free-trial/
# - a free academic license: https://www.gurobi.com/academia/academic-program-and-licenses/
#
# Without a license, Gurobi runs in restricted mode. In this case the notebook
# will only solve a reduced toy instance (2 days, 10 stations).

import gurobipy as gp

Licence case

In [ ]:
#no licence:
mini_sample = True #(toy model)

#I've got my licence:
mini_sample = False

datadir = "mini_data/" if mini_sample else "data/"

In [ ]:
from pathlib import Path
import ast

import numpy as np
import pandas as pd

from planvisit import Weekplan
from planrout import TruckRoutes
from visualizer import TruckRoutesVisualizer

## Configuration

The pipeline behaviour can be controlled through the following parameters.

In [ ]:
CONFIG = {
    "frontiers": datadir + "frontiers.csv",
    "stations": datadir + "stations.csv",
    "distances": datadir + "distance_km.npy",

    "operation_per_day": 100,
    "penalty_same_type": 5,

    "output_dir": Path("optim_output"),

    # Solver configuration
    "solve": "fast",  # "fast" (≈5min) or "best" (≈10min)

    # Optional visualization
    "visualize": True,
}

## Load optimization data

We load two datasets:

• **Frontier strategies**: strategic objectives for each station  
• **Geographic attributes**: latitude / longitude of stations

Distances between stations are computed using the **Haversine distance**.

In [ ]:
def load_optimization_data():
    if mini_sample:
        print("Mini sample activated")

    frontiers = pd.read_csv(CONFIG["frontiers"])
    stations = pd.read_csv(CONFIG["stations"])
    distances = np.load(CONFIG["distances"])

    return frontiers, stations, distances

Let's load the data and inspect the station table.

In [ ]:
frontiers, stations_all, dist_matrix = load_optimization_data()

Mini sample activated


In [ ]:
frontiers.head()

,station,sign,frontiere_bas,frontiere_haut
0,16115,15,"['[1100000]', '[0000101]', '[1000011]', '[0011...",['[1111111]']
1,42707,15,"['[0000010]', '[0001000]', '[1000000]', '[0100...",['[1111111]']
2,20117,15,"['[0000101]', '[1011000]', '[0000011]', '[0010...",['[1111111]']
3,15045,15,"['[1010000]', '[1100000]', '[1001000]', '[0000...",['[1111111]']
4,13008,15,"['[1100101]', '[1001111]', '[0111100]', '[1110...",['[1111111]']


In [ ]:
stations_all.head()

,station,sign,station_code,latitude,longitude
0,16115,15,16115,48.852587,2.262925
1,42707,15,42707,48.812405,2.361709
2,20117,15,20117,48.872667,2.407913
3,15045,15,15045,48.828052,2.292428
4,13008,15,13008,48.831909,2.354694


## Convert raw data into optimization parameters

The solver requires structured inputs:

• station sets  
• admissible strategies  
• routing costs  
• planning horizon

In [ ]:
def prepare_optimization_params(
    frontiers,
    stations_all,
    distance_matrix
):

    def parse_frontiers_logic(cell):

        if not isinstance(cell, str):
            return cell

        raw_list = ast.literal_eval(cell)

        limit = 2 if mini_sample else 7

        return [list(map(int, x.strip("[]")))[:limit] for x in raw_list]

    params = {"vide": {}, "plein": {}, "routing": {}}

    nin_limit = CONFIG["operation_per_day"] // 2

    for sens, sign_val in [("vide", 15), ("plein", -15)]:

        sub = frontiers[frontiers["sign"] == sign_val]

        params[sens] = {
            "station_ids": sub["station"].tolist(),
            "strategies": {
                "down": sub["frontiere_bas"].apply(parse_frontiers_logic).tolist(),
                "up": sub["frontiere_haut"].apply(parse_frontiers_logic).tolist(),
            },
            "Nin": nin_limit,
            "active_mask": (frontiers["sign"] == sign_val).values,
            "losses": 0,
        }

    params["routing"]["distance_matrix"] = distance_matrix
    params["routing"]["station_ids_global"] = stations_all["station_code"].tolist()
    params["routing"]["penalty_same_type"] = CONFIG["penalty_same_type"]

    try:
        sample = params["vide"]["strategies"]["down"][0][0]
        n_days = len(sample)
    except:
        n_days = 7

    dims = {
        "S_vide": range(len(params["vide"]["station_ids"])),
        "S_plein": range(len(params["plein"]["station_ids"])),
        "N": range(n_days),
    }

    return dims, params

In [ ]:
dims, params = prepare_optimization_params(
    frontiers,
    stations_all,
    dist_matrix
)
if mini_sample:
    print("solving MINI sample case (toy model):")
else:
    print("solving FULL model:")
print(dims)

solving MINI sample case (toy model):
{'S_vide': range(0, 5), 'S_plein': range(0, 5), 'N': range(0, 2)}


## Solve routing model

We solve the truck routing problem using the `TruckRoutes` optimizer.

⚠️  
If you get GurobiError: Model too large for size-limited license, rerun notebook with MINI_SAMPLE=TRUE (toy model)

In [ ]:
n_models = 3 if CONFIG["solve"] == "best" else 2

routs = TruckRoutes(
    dims,
    params,
    verbose=True,
    nmodels=n_models,
    solve=CONFIG["solve"],
)

routs.solve(0)
print("================")
print("First solving step achieved! let improve solution")
print("================")

for m in range(1, n_models):

    print(f"Solving model {m+1}/{n_models}")
    routs.solve(m)

Set parameter Method to value 3
Set parameter TimeLimit to value 60
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 3 3250U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Non-default parameters:
TimeLimit  60
Method  3

Optimize a model with 894 rows, 907 columns and 3118 nonzeros
Model fingerprint: 0x61178d6b
Variable types: 665 continuous, 242 integer (242 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [5e-01, 2e+01]
  Bounds range     [1e+00, 1e+01]
  RHS range        [2e+00, 5e+01]
Using partitions.
Found heuristic solution: objective -8.0000000
Presolve removed 449 rows and 304 columns
Presolve time: 0.05s
Presolved: 445 rows, 603 columns, 2185 nonzeros
Found heuristic solution: objective 82.0000000
Variable types: 360 continuous, 243 integer (243 binary)

Use crossover to convert LP symmetric solution to

In [ ]:
## Export optimized truck routes

In [ ]:
output_dir = CONFIG["output_dir"]
output_dir.mkdir(exist_ok=True)

output_file = output_dir / "planning_camions_final.csv"

routs.to_csv(output_file, id_model=n_models - 1)

print("Results saved to:", output_file)

CSV exporté vers optim_output\planning_camions_final.csv
Results saved to: optim_output\planning_camions_final.csv


In [ ]:
## Visualize routes

In [ ]:
if CONFIG["visualize"]:

    vis = TruckRoutesVisualizer(routs, stations_all)

    vis.extract_chains(m=n_models - 1)

    vis.save_routes_to_txt(
        m=n_models - 1,
        output_dir=output_dir,
    )

    vis.plot_routes(
        m=n_models - 1,
        output_dir=output_dir,
    )

    -> Plan textuel sauvegardé : optim_output\plan_regulation.txt
    -> Carte sauvegardée : optim_output\tournee_jour_0.png
    -> Carte sauvegardée : optim_output\tournee_jour_1.png
